# Section 2 — Missing Value Identification and Basic Imputation (20 pts)

## Learning Objectives
- Replace `-200` sentinel values with NaN
- Compute missing-value counts and percentages per column
- Apply at least two basic strategies: removal + mean/median imputation

## Your Decisions Log

| Step | Decision | Evidence (plot / table / stat) |
|------|----------|--------------------------------|
|      |          |                                |

## Required Formulas
- Missing % = (missing count / total rows) × 100
- Mean imputation: x̂ = x̄
- Median imputation: x̂ = median(X)

## Tasks
- a) Replace -200 with NaN
- b) Calculate missing count and percentage per column
- c) Decide keep / remove / impute for each column
- d) Apply removal + mean or median imputation
- e) Justify each decision with a clear criterion

In [16]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display

from src.config import RAW_DATA_PATH, MISSING_SENTINEL, MISSINGNESS_THRESHOLD, FIGURES_DIR
from src.load_data import load_raw_air_quality, build_datetime
from src.missing_values import (
    replace_sentinel_with_nan,
    missing_summary,
    decide_column_action,
    impute_basic,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Start from the raw dataset and rebuild the datetime column so this notebook can run on its own.
df = load_raw_air_quality(RAW_DATA_PATH)
df = build_datetime(df)
print("=" * 72)
print("DATETIME COLUMN MODIFICATION")
print("=" * 72)
display(df[['Date', 'Time', 'Datetime']].head())

DATETIME COLUMN MODIFICATION


,Date,Time,Datetime
0,10/03/2004,18.00.00,2004-03-10 18:00:00
1,10/03/2004,19.00.00,2004-03-10 19:00:00
2,10/03/2004,20.00.00,2004-03-10 20:00:00
3,10/03/2004,21.00.00,2004-03-10 21:00:00
4,10/03/2004,22.00.00,2004-03-10 22:00:00


In [17]:
# Replace the dataset's sentinel value with proper missing values.
df = replace_sentinel_with_nan(df, sentinel=MISSING_SENTINEL)
print('Sentinel values replaced with NaN.')
print(df.isna().sum().sort_values(ascending=False).head(10).to_string())

NotImplementedError: Implement sentinel replacement (Section 2a)

In [ ]:
# Summarize missingness and sort the columns from most affected to least affected.
missing_df = missing_summary(df).sort_values('missing_pct', ascending=False).reset_index(drop=True)
display(missing_df)

SyntaxError: invalid syntax (1422113774.py, line 5)

In [ ]:
# Translate missingness into a simple keep/remove/impute decision per column.
missing_df['action'] = missing_df['missing_pct'].apply(
    lambda pct: decide_column_action(pct, MISSINGNESS_THRESHOLD)
)
display(missing_df[['column', 'missing_count', 'missing_pct', 'action']])

In [ ]:
# Drop the columns with excessive missingness, remove rows without a usable datetime, and impute the rest with the median.
columns_to_remove = missing_df.loc[missing_df['action'] == 'remove', 'column'].tolist()
columns_to_impute = missing_df.loc[missing_df['action'] == 'impute', 'column'].tolist()
columns_to_impute = [column for column in columns_to_impute if column not in {'Date', 'Time', 'Datetime'}]

df_clean = df.dropna(subset=['Datetime']).drop(columns=columns_to_remove)
df_clean = impute_basic(df_clean, columns=columns_to_impute, strategy='median')

print('Removed columns:', columns_to_remove)
print('Imputed columns:', columns_to_impute)
display(df_clean.head())

In [ ]:
# Visualize the missing-value profile so the decision threshold is easy to justify.
plt.figure(figsize=(12, 6))
sns.barplot(data=missing_df, x='column', y='missing_pct', color='#4c72b0')
plt.axhline(MISSINGNESS_THRESHOLD * 100, color='tomato', linestyle='--', label='Threshold')
plt.xticks(rotation=75, ha='right')
plt.ylabel('Missing %')
plt.xlabel('Column')
plt.title('Missingness by Column')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'missingness_by_column.png', dpi=150, bbox_inches='tight')
plt.show()

## Guiding Questions

1. **What threshold did you use to decide whether a column had excessive missingness?**

   I used a 40% missingness threshold. Columns above that cutoff were treated as excessive and removed; columns below it were kept and either imputed or left unchanged depending on their role in the analysis.

2. **Why is the median sometimes preferable to the mean when imputing numerical variables?**

   The median is more robust to skewed distributions and extreme values. For air-quality data, where occasional spikes and outliers are common, the median usually gives a more representative central value than the mean.